In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer, losses
import warnings
import random
import os
from tqdm import tqdm
warnings.filterwarnings("ignore")

# Train model

In [2]:
sent_path = '../data/training_set_us.csv'

### Prepare model

In [4]:
# #model
model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-mpnet-base-v2")

#dataset and dataloader
train_sentences = random.sample(pd.read_csv(sent_path)['sentence'].tolist(), 400000)

train_dataloader = losses.ContrastiveTensionDataLoader(
    train_sentences,
    batch_size=12,
    pos_neg_ratio=3
)

#loss
train_loss = losses.ContrastiveTensionLoss(model=model)

### Trainer

In [5]:
os.environ["OMP_NUM_THREADS"] = "8"
os.environ["MKL_NUM_THREADS"] = "8"
os.environ["NUMEXPR_NUM_THREADS"] = "8"

In [6]:
import torch
torch.set_num_threads(8)

In [8]:
model.max_seq_length = 128

model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=1,
    warmup_steps=600,
    optimizer_params={"lr": 2e-5},
    show_progress_bar=True
)

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
500,3.974700
1000,0.452500
1500,0.457500
2000,0.348300
2500,0.361700
3000,0.286300
3500,0.335600
4000,0.302900
4500,0.337100
5000,0.317200


In [9]:
#save model
model.save('../outputs/doc_embedding_model_us')

# Encode documents

In [6]:
#load model
model = SentenceTransformer(
    "../outputs/doc_embedding_model_us",
    tokenizer_kwargs={"fix_mistral_regex": True}
)

In [9]:
docs = pd.read_csv('../outputs/interventions_us.csv', index_col=0)

In [ ]:
import torch
torch.set_num_threads(12)

In [18]:
ints = docs["speech"].tolist()

doc_embeddings = model.encode(
    ints,
    batch_size=16,      
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

Batches:   0%|          | 0/51425 [00:00<?, ?it/s]

In [ ]:
docs['embedding'] = doc_embeddings.tolist()
docs.to_csv('../outputs/interventions_us_with_embeddings.csv')

: 

# Repeat with colombian docs